# Autoencoder for Anomaly Detection (exploratory)
# Not being actively developed

Trained on **non-recombinant sequences only**. The hope is that
breakpoint regions in recombinant sequences will be poorly reconstructed,
producing a complementary anomaly signal.

**Caveat**: the current AE has structural problems (global pooling
destroys positional information; trained on `[parent, parent, parent]`
but applied to `[recomb, p1, p2]`). It is kept here as a starting point
but is **not** part of the primary detection pipeline. See `TODO.md`.

Setup cells (imports, config, data loading) are duplicated from `CNN.ipynb`
so this notebook can be run independently.


## Setup and data loading

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path
import os
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# TensorFlow/Keras
from tensorflow.keras.layers import (
    Input, Conv1D, MaxPooling1D, UpSampling1D,
    Dropout, BatchNormalization, Dense, Flatten,
    Concatenate, GlobalAveragePooling1D, Reshape,
    Add, Activation
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam, AdamW
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras import backend as K

# Scikit-learn
from sklearn.metrics import (
    precision_recall_fscore_support, roc_auc_score,
    roc_curve, auc, precision_recall_curve, average_precision_score
)

# SciPy (peak detection for breakpoint evaluation)
from scipy.signal import find_peaks

# BioPython for FASTA parsing
from Bio import SeqIO

# Progress tracking (tqdm.auto falls back to plain text bar if widgets unavailable)
from tqdm.auto import tqdm

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# Apple Silicon / Metal sanity check
# Confirms TensorFlow can actually run ops on the integrated GPU,
# not just *see* one. On macOS Silicon you need tensorflow-metal:
#     pip install tensorflow-metal

import platform

print(f"Platform: {platform.platform()}  ({platform.machine()})")
print(f"TensorFlow: {tf.__version__}")

gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs visible to TF: {gpus}")

if not gpus:
    print("\n[!] No GPU detected. On Apple Silicon, run:")
    print("        pip install tensorflow-metal")
    print("    and restart the kernel.")
else:
    # Force a real op onto the GPU and check where it ran. TF silently
    # falls back to CPU for unsupported ops, so a non-empty GPU list is
    # not a sufficient signal on its own.
    with tf.device('/GPU:0'):
        a = tf.random.normal([4096, 4096])
        b = tf.matmul(a, a)
    print(f"matmul executed on: {b.device}")
    if 'GPU' in b.device:
        print("[OK] GPU acceleration is active.")
    else:
        print("[!] Op fell back to CPU despite GPU being visible.")

In [ ]:
# Configuration

# Data paths
DATA_ROOT = Path("dataRaw")
TRAIN_DIRS = ["XML-1", "XML-2", "XML-3", "XML-4", "XML-5"]
TEST_DIR = "UnseenTestSet"

# Sequence encoding
MAX_SEQ_LEN = 4000          # Pad/truncate all sequences to this length
NUCLEOTIDES = ['A', 'T', 'G', 'C', '-']
N_CHANNELS = len(NUCLEOTIDES)  # 5 (one-hot dimension per sequence)
# Triplet one-hot (15) + 3 comparison channels (match_p1, match_p2, informative)
N_INPUT_CHANNELS = 3 * N_CHANNELS + 3  # 18

# Label generation
BP_WINDOW = 10              # +/- bp around each breakpoint edge for labels
TOLERANCE = 200             # +/- bp tolerance for evaluation

# Training
BATCH_SIZE = 16
EPOCHS = 100
LR = 1e-4
VAL_SPLIT = 0.15

# Focal loss
FOCAL_ALPHA = 0.25
FOCAL_GAMMA = 2.0

print(f"Max sequence length: {MAX_SEQ_LEN}")
print(f"Input channels (triplet + comparisons): {N_INPUT_CHANNELS}")
print(f"Training directories: {TRAIN_DIRS}")

In [ ]:
def one_hot_encode(sequence, max_length=MAX_SEQ_LEN):
    """One-hot encode a nucleotide sequence (A/T/G/C/gap) with padding.

    Args:
        sequence: Nucleotide string.
        max_length: Pad or truncate to this length.

    Returns:
        np.ndarray of shape (max_length, 5).
    """
    nuc_idx = {'A': 0, 'T': 1, 'G': 2, 'C': 3, '-': 4}
    encoded = np.zeros((max_length, N_CHANNELS), dtype=np.float32)
    for i, nuc in enumerate(sequence[:max_length].upper()):
        idx = nuc_idx.get(nuc, 4)  # Unknown nucleotides treated as gaps
        encoded[i, idx] = 1.0
    return encoded


def _seq_to_index(sequence, max_length=MAX_SEQ_LEN):
    """Per-position nucleotide index, with -1 marking padding (post sequence end)."""
    nuc_idx = {'A': 0, 'T': 1, 'G': 2, 'C': 3, '-': 4}
    idx = np.full(max_length, -1, dtype=np.int16)
    for i, nuc in enumerate(sequence[:max_length].upper()):
        idx[i] = nuc_idx.get(nuc, 4)
    return idx


def encode_triplet(seq_recomb, seq_parent1, seq_parent2):
    """Encode a triplet with one-hot + explicit parent-comparison channels.

    The comparison channels expose the recombination signal directly:
    a breakpoint shows up as a flip in which parent the recombinant
    matches. Without these channels the conv stack must rediscover
    cross-channel comparison from scratch.

    Channels (shape (L, 18)):
        0-4    recombinant one-hot
        5-9    parent 1 one-hot
        10-14  parent 2 one-hot
        15     match_p1: recombinant base equals parent 1 base
        16     match_p2: recombinant base equals parent 2 base
        17     informative: parent 1 base differs from parent 2 base

    All comparison channels are zero at padding positions (past the
    end of any of the three sequences), so they do not add spurious
    signal where there is no data.
    """
    enc_r = one_hot_encode(seq_recomb)
    enc_1 = one_hot_encode(seq_parent1)
    enc_2 = one_hot_encode(seq_parent2)

    r  = _seq_to_index(seq_recomb)
    p1 = _seq_to_index(seq_parent1)
    p2 = _seq_to_index(seq_parent2)
    valid = (r >= 0) & (p1 >= 0) & (p2 >= 0)

    match_p1   = ((r  == p1) & valid).astype(np.float32)[:, None]
    match_p2   = ((r  == p2) & valid).astype(np.float32)[:, None]
    informative = ((p1 != p2) & valid).astype(np.float32)[:, None]

    return np.concatenate(
        [enc_r, enc_1, enc_2, match_p1, match_p2, informative],
        axis=1,
    )


# Sanity checks
test_enc = one_hot_encode("ATGC-N", max_length=6)
print(f"One-hot shape: {test_enc.shape}")
print(f"A=[1,0,0,0,0]: {test_enc[0].tolist()}")
print(f"T=[0,1,0,0,0]: {test_enc[1].tolist()}")
print(f"Gap=[-]:       {test_enc[4].tolist()}")
print(f"Unknown->gap:  {test_enc[5].tolist()}")

trip = encode_triplet("ATGCAT", "ATGGGT", "ATCCAT")
print(f"\nTriplet shape: {trip.shape}  (expected (4000, {N_INPUT_CHANNELS}))")
# Position 2: r=G, p1=G, p2=C  -> match_p1=1, match_p2=0, informative=1
print(f"Pos 2 comparison [m_p1, m_p2, info]: {trip[2, 15:].tolist()}")
# Position 3: r=C, p1=G, p2=C  -> match_p1=0, match_p2=1, informative=1
print(f"Pos 3 comparison [m_p1, m_p2, info]: {trip[3, 15:].tolist()}")
# Position 0: r=A, p1=A, p2=A  -> match_p1=1, match_p2=1, informative=0
print(f"Pos 0 comparison [m_p1, m_p2, info]: {trip[0, 15:].tolist()}")

In [ ]:
def generate_labels(bp_start, bp_end, seq_length=MAX_SEQ_LEN,
                    mode='breakpoint', window=BP_WINDOW):
    """Generate per-position binary labels from breakpoint coordinates.

    Args:
        bp_start: Breakpoint start position.
        bp_end: Breakpoint end position.
        seq_length: Length of the label vector.
        mode: 'breakpoint' marks only edges (+/- window), 'region' marks the
              entire recombinant region between breakpoints.
        window: Half-width of the window around each breakpoint edge.

    Returns:
        np.ndarray of shape (seq_length,) with binary labels.
    """
    labels = np.zeros(seq_length, dtype=np.float32)
    circular = bp_start > bp_end  # Circular genome wrapping

    if mode == 'region':
        if circular:
            labels[bp_start:] = 1.0
            labels[:bp_end + 1] = 1.0
        else:
            labels[bp_start:bp_end + 1] = 1.0
    else:  # breakpoint-only
        for bp in [bp_start, bp_end]:
            lo = max(0, bp - window)
            hi = min(seq_length, bp + window + 1)
            labels[lo:hi] = 1.0

    return labels


# Verify both modes
lbl_bp = generate_labels(500, 1500, mode='breakpoint')
lbl_rg = generate_labels(500, 1500, mode='region')
lbl_circ = generate_labels(3500, 200, mode='breakpoint')
print(f"Breakpoint-only positives: {int(lbl_bp.sum())}")
print(f"Region positives:          {int(lbl_rg.sum())}")
print(f"Circular breakpoint:       {int(lbl_circ.sum())}")

In [ ]:
def parse_simulation(fasta_path):
    """Parse one SANTA simulation run (FASTA + associated CSVs).

    For each recombination event the function extracts the recombinant
    sequence and two parent sequences, encodes the triplet, and
    generates per-position labels.

    Args:
        fasta_path: Path to the .fa alignment file.

    Returns:
        List of dicts with keys 'input', 'labels_bp', 'labels_region', 'meta'.
    """
    fasta_path = Path(fasta_path)
    sim_csv = fasta_path.parent / f"{fasta_path.name}SimVSRealCompare.csv"
    stats_csv = fasta_path.parent / f"{fasta_path.name}RecombIdentifyStats.csv"

    if not sim_csv.exists() or not stats_csv.exists():
        return []

    # FASTA IDs are integers in this dataset
    try:
        seqs = {int(r.id): str(r.seq) for r in SeqIO.parse(fasta_path, 'fasta')}
    except Exception:
        return []

    try:
        sim = pd.read_csv(sim_csv, skipinitialspace=True)
        stats = pd.read_csv(stats_csv, skipinitialspace=True)
    except Exception:
        return []

    results = []
    for _, row in sim.iterrows():
        event = row['RDPEvent']
        recomb_id = int(row['ActualRecomb'])
        bp_start = int(row['SimBPStart'])
        bp_end = int(row['SimBPEnd'])

        # Each event has 3 hypothesis rows in the stats CSV. Skip events
        # that don't follow this structure (data corruption, partial run).
        ev_rows = stats[stats['Event'] == event]
        if len(ev_rows) != 3:
            continue

        # Identify parents: the two hypothesis rows whose ISeqs(A) does
        # NOT contain the actual recombinant ID. ISeqs(A) is a $-delimited
        # list of seq IDs; we take the first valid integer from each.
        parent_ids = []
        for _, sr in ev_rows.iterrows():
            ids = [int(s.strip()) for s in str(sr['ISeqs(A)']).split('$')
                   if s.strip().isdigit()]
            if recomb_id in ids:
                continue
            if ids:
                parent_ids.append(ids[0])

        if len(parent_ids) < 2:
            continue

        if not all(sid in seqs for sid in [recomb_id, parent_ids[0], parent_ids[1]]):
            continue

        triplet = encode_triplet(
            seqs[recomb_id], seqs[parent_ids[0]], seqs[parent_ids[1]]
        )

        results.append({
            'input': triplet,
            'labels_bp': generate_labels(bp_start, bp_end, mode='breakpoint'),
            'labels_region': generate_labels(bp_start, bp_end, mode='region'),
            'meta': {
                'file': fasta_path.name,
                'event': event,
                'recomb_id': recomb_id,
                'parent1_id': parent_ids[0],
                'parent2_id': parent_ids[1],
                'bp_start': bp_start,
                'bp_end': bp_end,
                # rstrip trailing gaps; internal gaps are kept (they are
                # alignment artefacts and the breakpoint coordinates are
                # in alignment space, not raw-sequence space).
                'actual_len': len(seqs[recomb_id].rstrip('-')),
            },
        })

    return results


# Smoke test on one file
sample_fa = sorted((DATA_ROOT / "XML-1").glob("*.fa"))[0]
sample_triplets = parse_simulation(sample_fa)
print(f"Parsed {len(sample_triplets)} triplets from {sample_fa.name}")
if sample_triplets:
    print(f"Input shape:  {sample_triplets[0]['input'].shape}")
    print(f"Labels shape: {sample_triplets[0]['labels_bp'].shape}")
    print(f"Metadata:     {sample_triplets[0]['meta']}")

In [ ]:
def load_dataset(directories, label_mode='breakpoint', max_files=None):
    """Load and aggregate triplet data from multiple simulation directories.

    Args:
        directories: List of subdirectory names under DATA_ROOT.
        label_mode: 'breakpoint' or 'region'.
        max_files: Optional per-directory file limit (for quick testing).

    Returns:
        X  -- np.ndarray (n, MAX_SEQ_LEN, 15)
        y  -- np.ndarray (n, MAX_SEQ_LEN)
        meta -- list of metadata dicts
    """
    inputs, labels, meta = [], [], []

    for d in directories:
        fa_files = sorted((DATA_ROOT / d).glob("*.fa"))
        if max_files:
            fa_files = fa_files[:max_files]

        print(f"\nProcessing {d}: {len(fa_files)} files")
        for fa in tqdm(fa_files, desc=d):
            for t in parse_simulation(fa):
                inputs.append(t['input'])
                key = 'labels_bp' if label_mode == 'breakpoint' else 'labels_region'
                labels.append(t[key])
                meta.append(t['meta'])

    X = np.array(inputs, dtype=np.float32)
    y = np.array(labels, dtype=np.float32)

    pos_frac = np.sum(y) / y.size * 100
    print(f"\n{'='*60}")
    print(f"Loaded {X.shape[0]} samples")
    print(f"X shape: {X.shape}  |  y shape: {y.shape}")
    print(f"Positive labels: {np.sum(y):.0f} / {y.size} ({pos_frac:.3f}%)")
    print(f"{'='*60}")

    return X, y, meta

In [ ]:
# Load training data
# Set max_files=10 for a quick test run; set to None for the full dataset
X_all, y_all, meta_all = load_dataset(
    TRAIN_DIRS,
    label_mode='breakpoint',
    max_files=100,
)

In [ ]:
# Train / validation split — GROUPED BY FASTA FILE
# All events within a .fa file come from the same SANTA simulation
# (shared population, parents, mutation params), so they are not i.i.d.
# Splitting at the event level lets correlated samples appear in both
# train and val. Group-shuffle by file to prevent that leakage.

files = np.array([m['file'] for m in meta_all])
unique_files = np.unique(files)

rng = np.random.default_rng(42)
rng.shuffle(unique_files)

n_val_files = max(1, int(round(len(unique_files) * VAL_SPLIT)))
val_files = set(unique_files[:n_val_files])
val_mask = np.array([f in val_files for f in files])
train_mask = ~val_mask

X_train, y_train = X_all[train_mask], y_all[train_mask]
X_val,   y_val   = X_all[val_mask],   y_all[val_mask]
meta_train = [m for m, v in zip(meta_all, val_mask) if not v]
meta_val   = [m for m, v in zip(meta_all, val_mask) if v]

print(f"Files: {len(unique_files)} total  ->  "
      f"{len(unique_files) - n_val_files} train / {n_val_files} val")
print(f"Training:   {X_train.shape[0]} samples")
print(f"Validation: {X_val.shape[0]} samples")

# Sanity: no file appears on both sides
assert not (set(m['file'] for m in meta_train) & set(m['file'] for m in meta_val))

## Autoencoder model

In [ ]:
def build_autoencoder(input_shape=(MAX_SEQ_LEN, 3 * N_CHANNELS),
                      latent_dim=64, dropout=0.2):
    """Convolutional autoencoder for sequence reconstruction.

    Takes the 15-channel one-hot triplet only (no comparison channels) so
    that bumping N_INPUT_CHANNELS for the CNN does not break the AE.

    Encoder downsamples via strided convolutions; decoder upsamples back.
    Trained on non-recombinant sequences so breakpoint regions produce
    higher reconstruction error.

    Args:
        input_shape: (sequence_length, channels).
        latent_dim: Dimensionality of the bottleneck.
        dropout: Dropout rate.

    Returns:
        (encoder, decoder, autoencoder) Keras Models.
    """
    n_ch = input_shape[-1]

    # Encoder
    enc_in = Input(shape=input_shape, name='enc_input')

    x = Conv1D(64, 7, strides=2, padding='same', activation='relu')(enc_in)
    x = BatchNormalization()(x)
    x = Dropout(dropout)(x)

    x = Conv1D(128, 5, strides=2, padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(dropout)(x)

    x = Conv1D(256, 3, strides=2, padding='same', activation='relu')(x)
    x = BatchNormalization()(x)

    x = GlobalAveragePooling1D()(x)
    latent = Dense(latent_dim, activation='relu', name='latent')(x)

    encoder = Model(enc_in, latent, name='Encoder')

    # Decoder
    dec_in = Input(shape=(latent_dim,), name='dec_input')

    x = Dense(500 * 256, activation='relu')(dec_in)  # 4000 / 8 = 500
    x = Reshape((500, 256))(x)

    x = Conv1D(256, 3, padding='same', activation='relu')(x)
    x = UpSampling1D(2)(x)   # 500 -> 1000
    x = BatchNormalization()(x)

    x = Conv1D(128, 5, padding='same', activation='relu')(x)
    x = UpSampling1D(2)(x)   # 1000 -> 2000
    x = BatchNormalization()(x)

    x = Conv1D(64, 7, padding='same', activation='relu')(x)
    x = UpSampling1D(2)(x)   # 2000 -> 4000
    x = BatchNormalization()(x)

    dec_out = Conv1D(n_ch, 3, padding='same',
                     activation='sigmoid', name='reconstruction')(x)

    decoder = Model(dec_in, dec_out, name='Decoder')

    # Full autoencoder
    ae_out = decoder(encoder(enc_in))
    autoencoder = Model(enc_in, ae_out, name='Autoencoder')

    return encoder, decoder, autoencoder


encoder, decoder, autoencoder = build_autoencoder()
autoencoder.compile(optimizer=Adam(learning_rate=LR), loss='mse')
autoencoder.summary()

In [ ]:
def load_parent_sequences(directories, max_files=None):
    """Load non-recombinant sequences for autoencoder training.

    For each FASTA, every sequence whose ID is not listed as
    `ActualRecomb` for any event is treated as non-recombinant.

    NOTE: each parent sequence is replicated across the 3 triplet
    channel groups (so training input is `[parent, parent, parent]`).
    This is a known design issue -- see TODO.md item "Autoencoder
    redesign" -- it creates a distribution shift at inference, where
    the AE is applied to actual triplets `[recomb, p1, p2]`.

    Returns:
        np.ndarray of shape (n, MAX_SEQ_LEN, 3 * N_CHANNELS).
    """
    parent_seqs = []

    for d in directories:
        fa_files = sorted((DATA_ROOT / d).glob("*.fa"))
        if max_files:
            fa_files = fa_files[:max_files]

        print(f"\nCollecting parents from {d}: {len(fa_files)} files")
        for fa in tqdm(fa_files, desc=d):
            sim_csv = fa.parent / f"{fa.name}SimVSRealCompare.csv"
            if not sim_csv.exists():
                continue

            try:
                seqs = {int(r.id): str(r.seq) for r in SeqIO.parse(fa, 'fasta')}
                sim = pd.read_csv(sim_csv, skipinitialspace=True)
            except Exception:
                continue

            recomb_ids = set(sim['ActualRecomb'].astype(int).tolist())
            parent_ids = [sid for sid in seqs if sid not in recomb_ids]

            # Cap at 10 per file to keep dataset size manageable; the AE
            # benefits from diversity across files more than depth per file.
            for pid in parent_ids[:10]:
                enc = one_hot_encode(seqs[pid])
                triplet = np.concatenate([enc, enc, enc], axis=1)
                parent_seqs.append(triplet)

    X = np.array(parent_seqs, dtype=np.float32)
    print(f"\nLoaded {X.shape[0]} parent sequences, shape: {X.shape}")
    return X


X_parents = load_parent_sequences(TRAIN_DIRS, max_files=None)

In [ ]:
# Train autoencoder on parent (non-recombinant) sequences
ae_callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1),
]

ae_history = autoencoder.fit(
    X_parents, X_parents,  # Reconstruct input
    validation_split=0.15,
    batch_size=BATCH_SIZE,
    epochs=50,
    callbacks=ae_callbacks,
    verbose=1,
)

print("\nAutoencoder training complete.")

### Reconstruction error visualisation

Per-position MSE between input and reconstruction, with true breakpoint
positions overlaid in red. With the current AE design, do not expect a
clean spike at the breakpoint -- see the caveat above and TODO.md.

In [ ]:
def plot_reconstruction_error(sample_idx, X_data, y_labels, meta_list):
    """Visualise per-position reconstruction error alongside true breakpoints."""
    sample = X_data[sample_idx:sample_idx + 1]
    reconstruction = autoencoder.predict(sample, verbose=0)
    mse_per_pos = np.mean((sample[0] - reconstruction[0]) ** 2, axis=1)

    m = meta_list[sample_idx]
    L = m['actual_len']

    fig, axes = plt.subplots(2, 1, figsize=(16, 6), sharex=True)

    # Reconstruction error
    axes[0].plot(range(L), mse_per_pos[:L], color='purple', linewidth=1)
    axes[0].set_ylabel('Reconstruction MSE')
    axes[0].set_title(
        f'Autoencoder Reconstruction Error \u2014 {m["file"]} (Event {m["event"]})',
        fontweight='bold'
    )
    axes[0].grid(True, alpha=0.3)

    # Mark true breakpoints
    for bp in [m['bp_start'], m['bp_end']]:
        if bp < L:
            axes[0].axvline(bp, color='red', linestyle='--', alpha=0.7, label='True BP')

    # Deduplicate legend
    handles, lbls = axes[0].get_legend_handles_labels()
    by_label = dict(zip(lbls, handles))
    axes[0].legend(by_label.values(), by_label.keys())

    # Ground truth labels
    axes[1].fill_between(range(L), 0, y_labels[sample_idx][:L],
                         color='green', alpha=0.5, label='Breakpoint region')
    axes[1].set_xlabel('Genomic Position (bp)')
    axes[1].set_ylabel('Label')
    axes[1].set_ylim(-0.1, 1.1)
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    return fig


# Plot for first few validation samples
for i in range(min(3, len(X_val))):
    fig = plot_reconstruction_error(i, X_val, y_val, meta_val)
    fig.savefig(f'figures/ae_recon_error_{i}.png', dpi=150, bbox_inches='tight')
    plt.show()